# The PyTorch ecosystem — what to reach for

_PyTorch (a complete course)_

**The course finale: the libraries around PyTorch, and when to use each.**

PyTorch is a small, sharp core — tensors and autograd — surrounded by a huge ecosystem. The skill is not memorizing every library; it is knowing the three layers and reaching for the right one: the core (raw control), high-level trainers (remove boilerplate at scale), and pretrained-model hubs (skip training entirely).

---

This notebook is a practice scaffold for the **The PyTorch ecosystem — what to reach for** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q lightning
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
# Colab setup: torch is preinstalled; install the framework layers.
!pip -q install lightning transformers

# =====================================================================
# (A) PyTorch Lightning — the hand-written loop, organized away.
# =====================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import lightning as L

torch.manual_seed(0)

# tiny synthetic 2-class dataset (same spirit as the training-loop lesson)
N, D = 1000, 8
y = torch.randint(0, 2, (N,))
X = torch.randn(2, D)[y] * 2.0 + torch.randn(N, D)
train_dl = DataLoader(TensorDataset(X[:800], y[:800]), batch_size=32, shuffle=True)
val_dl   = DataLoader(TensorDataset(X[800:], y[800:]), batch_size=64)

class LitClassifier(L.LightningModule):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(D, 32), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 2),                 # raw logits (no softmax!)
        )

    def forward(self, x):
        return self.net(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)     # expects logits + int labels
        self.log("train_loss", loss, prog_bar=True)
        return loss                            # Lightning calls backward()+step()

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        acc = (logits.argmax(1) == y).float().mean()
        self.log("val_acc", acc, prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-2)

# No epoch/batch loop, no .to(device), no zero_grad/backward/step in YOUR code.
# Flip on multi-GPU + mixed precision with flags: devices=4, precision="16-mixed".
trainer = L.Trainer(max_epochs=15, accelerator="auto", devices=1,
                    logger=False, enable_checkpointing=False)
trainer.fit(LitClassifier(), train_dl, val_dl)

# =====================================================================
# (B) Hugging Face — use a PRETRAINED model, often with no training.
# =====================================================================
from transformers import pipeline, AutoTokenizer, AutoModel

# High-level: one line loads model + tokenizer and runs the task.
clf = pipeline("sentiment-analysis")          # downloads a pretrained model
print(clf("PyTorch's ecosystem makes shipping models fast."))
# -> [{'label': 'POSITIVE', 'score': 0.99...}]

# Lower-level: load a backbone + tokenizer yourself to get embeddings.
tok = AutoTokenizer.from_pretrained("bert-base-uncased")
bert = AutoModel.from_pretrained("bert-base-uncased")
enc = tok("hello pytorch", return_tensors="pt")   # tokenizer = the model's own
with torch.no_grad():                              # inference: no graph
    out = bert(**enc)
print("sequence embedding shape:", out.last_hidden_state.shape)  # [1, seq, 768]


## Visualize the data & results

_For the same fine-tune task, how much code do you write in raw PyTorch vs. Lightning vs. Hugging Face?_

In [ ]:
import numpy as np

# Illustrative line-count breakdown for the SAME task (fine-tune + multi-GPU
# + AMP + logging), summed per stack. Components are plausible estimates;
# the point is the monotonic trend, not exact counts.
labels = ["Raw PyTorch", "PyTorch Lightning", "Hugging Face"]
#            data  model  loop  scale  logging  misc
raw   = np.array([18,   14,   40,   22,     12,    14])  # write everything
light = np.array([18,   14,   12,    0,      0,     6])  # Trainer owns loop/scale/log
hf    = np.array([ 6,    2,    0,    0,      0,     4])  # pretrained model, pipeline

vals = [int(raw.sum()), int(light.sum()), int(hf.sum())]
for name, v in zip(labels, vals):
    print(f"{name:18s} {v:3d} lines")
print("ratio raw:lightning:hf =",
      round(vals[0] / vals[2], 1), ": ",
      round(vals[1] / vals[2], 1), ": 1")
# -> Raw PyTorch        120 lines
#    PyTorch Lightning   50 lines
#    Hugging Face        12 lines


## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You need to fine-tune a well-known text classifier (a Transformer) on your labeled data, and you want it working by tomorrow. Which part of the ecosystem do you reach for, and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Notice the model already exists pretrained. — _A pretrained Transformer means you skip training from scratch — the expensive part._
- Pick the hub built for exactly this. — _Hugging Face transformers gives AutoModelForSequenceClassification.from_pretrained + a matching tokenizer + a Trainer._

**Answer:** Hugging Face. Load with from_pretrained, tokenize with the model's own tokenizer, and fine-tune with the Trainer — far less code than a raw loop, and you start from learned weights instead of random ones.

</details>

**Problem 2.** Your raw training script works on one GPU. You now have 4 GPUs and want multi-GPU training plus mixed precision, but you would rather not rewrite the loop with Distributed Data Parallel by hand. What is the lightest-touch move?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Identify what is reusable vs. what is plumbing. — _Your per-batch math (forward/loss/backward) stays; only the loop scaffolding and device/precision handling change._
- Move the per-batch logic into a framework's hook. — _In Lightning, put it in training_step + configure_optimizers, then set Trainer(devices=4, precision="16-mixed")._

**Answer:** Use PyTorch Lightning (or Hugging Face Accelerate for a thinner wrapper). Lightning's Trainer flags turn on multi-GPU (DDP) and AMP without you writing the distributed loop. Your math is unchanged.

</details>

**Problem 3.** A teammate insists on using Lightning and Hugging Face for everything and never writing a raw loop. Why is that risky for a learner?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall that these frameworks wrap the same five-step loop. — _They hide zero_grad/forward/loss/backward/step, not replace them._
- Imagine a subtle bug inside the hidden loop. — _A wrong tensor shape or a device mismatch surfaces as a cryptic framework error; without the fundamentals you cannot trace it._

**Answer:** Frameworks save time only after you understand the raw loop. Skip the fundamentals and every framework error is unfixable magic. Learn the loop first (pt-training-loop), then use Lightning/Hugging Face as conveniences — not crutches.

</details>